# Promedio de horas y millas por conductor

Consulta la tabla `timesheet` y guarda en CSV el promedio de horas y millas registradas por conductor.

In [1]:
import sqlite3
from pathlib import Path

import pandas as pd

activity_dir = next(
    (
        candidate
        for candidate in (
            Path.cwd(),
            Path.cwd().parent,
            Path.cwd() / "P105_drivers_chatgpt",
        )
        if (candidate / "data" / "timesheet.csv").is_file()
    ),
    None,
)
if activity_dir is None:
    raise FileNotFoundError("No se encontró data/timesheet.csv")

timesheet = pd.read_csv(activity_dir / "data" / "timesheet.csv")
with sqlite3.connect(":memory:") as connection:
    timesheet.to_sql("timesheet", connection, index=False, if_exists="replace")
    mean_timesheet = pd.read_sql_query(
        '''
        SELECT
            driverId,
            AVG("hours-logged") AS mean_hours_logged,
            AVG("miles-logged") AS mean_miles_logged
        FROM timesheet
        GROUP BY driverId
        ORDER BY driverId
        ''',
        connection,
    )

output_path = activity_dir / "submission" / "mean_timesheet_by_driver.csv"
mean_timesheet.to_csv(output_path, index=False)
mean_timesheet.head()

,driverId,mean_hours_logged,mean_miles_logged
0,10,62.153846,2829.807692
1,11,70.038462,3448.076923
2,12,50.750000,2614.653846
3,13,52.442308,2579.346154
4,14,53.480769,2627.384615
